In [ ]:
import importlib
import os
from pathlib import Path

import gnss_tools.signals.gps_l1ca as gps_l1ca
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming

plt.rcParams.update({'font.size': 16})

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_names = sorted(map(lambda fp: fp.name, collects_dir.iterdir()))
print("Available experiments:", ", ".join([str(name) for name in available_experiment_names]))
experiment_name = available_experiment_names[2]
experiment_dir = collects_dir / experiment_name
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

collect_id = metadata.collect_ids[2]
band_id = metadata.band_ids[0]

collect_config = metadata.collects[collect_id]
channel_id = collect_config.channel_config_id
band_config = metadata.band_configurations[band_id]
channel_config = metadata.channel_configurations[channel_id]

inter_freq_l1_hz = band_config.inter_freq
samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
collect_filepath = experiment_dir / collect_config.filename

In [ ]:
buffer_duration_ms = 40
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)
buffer_size_bytes = sample_streaming.compute_sample_array_size_bytes(
    buffer_size_samples, sample_params.bit_depth, sample_params.is_complex
)
byte_buffer = bytearray(buffer_size_bytes)
sample_buffer = np.zeros(buffer_size_samples, dtype=np.complex64)

with open(collect_filepath, "rb") as f:
    f.readinto(byte_buffer)

sample_streaming.convert_to_complex64_samples(
    byte_buffer,
    sample_buffer,
    sample_params
)
baseband_samples = np.zeros(buffer_size_samples, dtype=np.complex64)
sample_buffer_uptime_epoch_ms = 0.0
sample_streaming.mixdown_samples(
    sample_buffer,
    baseband_samples,
    samp_rate,
    0.0,
    inter_freq_l1_hz
)
baseband_samples -= np.mean(baseband_samples)

In [ ]:
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
# hist_bins = np.arange(-16, 17)
hist_bins = np.arange(-128, 128)
ax.hist(sample_buffer.real, bins=hist_bins, rwidth=0.5, color="r", align="left", label="Real")
ax.hist(sample_buffer.imag, bins=hist_bins, rwidth=0.5, color="b", align="mid", label="Imaginary")
ax.set_title("Histogram of Raw Samples")
ax.set_xlabel("Sample Value")
ax.set_ylabel("Count")
ax.legend()
ax.grid()
plt.show()

# NOTE: should see normal-looking distribution for raw samples.  But if samples are real, the imaginary component should be all zeros.

In [ ]:
# Plot Welch PSD estimate of the samples
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
freqs, psd_orig = scipy.signal.welch(
    sample_buffer,
    fs=samp_rate,
    nperseg=4096,
    noverlap=2048,
    window="hann",
    return_onesided=False,
    scaling="density",
)
_, psd = scipy.signal.welch(
    baseband_samples,
    fs=samp_rate,
    nperseg=4096,
    noverlap=2048,
    window="hann",
    return_onesided=False,
    scaling="density",
)
freqs = np.fft.fftshift(freqs)
psd_orig = np.fft.fftshift(psd_orig)
psd = np.fft.fftshift(psd)
# ax.plot(freqs/1e6, 10*np.log10(psd_orig), color="black")
ax.plot(freqs/1e6, 10*np.log10(psd_orig), color="gray")
ax.plot(freqs/1e6, 10*np.log10(psd), color="black")
ax.set_title("Welch PSD Estimate of Raw Samples")
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("Power/Frequency (dB/Hz)")
ax.grid()
plt.show()

In [ ]:
GPS_L1CA_acq_code_params = {
    f"G{prn:02}": bpsk_acquisition.AcqSignalCodeParameters(
        rate_chips_per_sec=gps_l1ca.CODE_RATE,
        length_chips=gps_l1ca.CODE_LENGTH,
        sequence=1 - 2 * gps_l1ca.get_GPS_L1CA_code_sequence(prn),
    ) for prn in range(1, 33)
}

acq_config = bpsk_acquisition.AcquisitionConfiguration(
    replica_duration_ms=4,
    num_blocks=8,
    sample_rate=samp_rate,
    min_search_doppler_hz=-5000,
    max_search_doppler_hz=5000,
)

# Compute bin probability of false alarm
# If we treat each delay/Doppler bin as single detection test
# individual probability of false alarm per bin p_fa_bin
# yields overall probability of false alarm p_fa_total
num_doppler_bins = acq_config.num_doppler_bins
num_code_phase_bins = acq_config.replica_length_samples
num_detection_tests = num_doppler_bins * num_code_phase_bins
print(f"Acq. num Doppler bins: {num_doppler_bins}")
print(f"Acq. num code phase bins: {num_code_phase_bins}")
print(f"Number of detection tests per signal: {num_detection_tests}")
p_fa_total = 1e-9
p_fa_bin = 1 - (1 - p_fa_total) ** (1 / num_detection_tests)
print(f"Desired overall P_fa: {p_fa_total:.2e}, per-bin P_fa: {p_fa_bin:.2e}")
print("")

acq_results = bpsk_acquisition.run_acquisition(
    baseband_samples,
    sample_buffer_uptime_epoch_ms,
    acq_config,
    GPS_L1CA_acq_code_params,
    prob_false_alaram=p_fa_bin,
    print_progress=True,
    noise_var_method="abscorrvar"
)
acquired_signals = sorted([acq_result.signal_id for acq_result in acq_results.values() if acq_result.signal_detected])

In [ ]:
# Plot peak amplitudes
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
all_sig_ids = sorted(acq_results.keys())
all_peak_vals = [
    acq_results[sig_id].normalized_peak_value
    for sig_id in all_sig_ids
]
ax.stem(range(len(all_peak_vals)), all_peak_vals, basefmt=" ")
detection_threshold = acq_results["G01"].detection_threshold
ax.plot([0, 33], [detection_threshold] * 2, color="red", linestyle="--", label="Detection Threshold")
ax.legend()
ax.set_yscale("log")
ax.set_xticks(range(len(all_sig_ids)))
ax.set_xticklabels(all_sig_ids, rotation=45)
ax.set_title("Acquisition Peak Correlation Values")
ax.set_xlabel("Signal ID (PRN)")
ax.set_ylabel("Normalized Peak Correlation Value")
ax.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
cmap = plt.get_cmap("tab20b")
for i, (signal_id, acq_result) in enumerate(acq_results.items()):
    color = cmap(i / 40.0)
    marker = ("o", "x")[i % 2]
    ax.plot(
        acq_result.correlation_result.doppler_bins_hz,
        acq_result.correlation_result.correlation_matrix[:, acq_result.peak_code_phase_bin],
        color=color,
        marker=marker,
        # label=f"{signal_id} (SNR={acq_result.acq_snr_dB:.1f} dB)",
        label=f"{signal_id}",
    )
ax.set_yscale("log")
ax.set_title("Acquisition Correlation Results")
ax.set_xlabel("Doppler Frequency [Hz]")
ax.set_ylabel("Peak Slice Correlation Magnitude")
ax.legend(ncol=4, fontsize=8)
ax.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=150)
ax = fig.add_subplot(111)
sig_id = sorted(acq_results.keys())[5]
acq_result = acq_results[sig_id]
acq_config = acq_result.config
correlation = acq_result.correlation_result.correlation_matrix
num_doppler_bins, num_code_phases = correlation.shape
extent = [0, num_code_phases, acq_config.min_search_doppler_hz, acq_config.max_search_doppler_hz]

peak_doppler_bin = acq_result.peak_doppler_bin
peak_doppler_hz = acq_result.correlation_result.doppler_bins_hz[peak_doppler_bin]
peak_code_phase_bin = acq_result.peak_code_phase_bin
print(f"Peak correlation at Doppler: {peak_doppler_hz} Hz, Code Phase: {peak_code_phase_bin} samples")

im = ax.imshow(
    correlation,
    extent=extent,
    aspect="auto",
    interpolation="nearest",
    cmap="plasma",
    origin="lower",
    # vmin=noise_power,
    vmin=0,
    # vmax=500
)
# ax.hlines(acquired_dopplers, 0, num_code_phases, colors="white", linestyles="--", label="Acquired Dopplers")
# code_phase_window = 30000  # samples
# code_phase_window = 2500  # samples
code_phase_window = 150  # samples
ax.set_xlim(peak_code_phase_bin - code_phase_window, peak_code_phase_bin + code_phase_window)
ax.set_xlabel("Code Phase [samples]")
ax.set_ylabel("Doppler Frequency [Hz]")
ax.set_title(f"Delay-Doppler Correlation Map for PRN {int(sig_id[1:])}")
plt.colorbar(im, label="Correlation Magnitude")
plt.show()

In [ ]:
sig_id = acquired_signals[0]
acq_result = acq_results[sig_id]
corr_matrix = acq_result.correlation_result.correlation_matrix

y_noise_mean = np.mean(corr_matrix)
sigma_n = np.sqrt(y_noise_mean / (2 * acq_config.num_blocks))
# sigma_n = np.std(sample_buffer)

# y_noise_var = np.var(corr_matrix)
# sigma_n = np.sqrt(np.sqrt(y_noise_var / (4 * acq_config.num_blocks)))


normalized_corr_matrix = corr_matrix / sigma_n**2
# hist_max_val = np.max(normalized_corr_matrix)
hist_max_val = 120

hist_bins = np.linspace(0, hist_max_val, 100)
hist = np.histogram(normalized_corr_matrix.flatten(), bins=hist_bins)[0]

# plot corr matrix histogram
fig = plt.figure(figsize=(8, 4), dpi=150)
# fig = plt.figure(figsize=(5, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
ax.bar(
    hist_bins[:-1],
    hist,
    width=hist_bins[1]-hist_bins[0],
    color="blue",
    alpha=0.7,
)
# add chi-squared distribution overlay
x_vals = np.linspace(0, np.max(normalized_corr_matrix), 1000)
chi2_pdf = scipy.stats.chi2.pdf(x_vals, df=2 * acq_config.num_blocks)
ax.plot(
    x_vals,
    chi2_pdf * np.max(hist) / np.max(chi2_pdf),
    color="red",
    linewidth=2,
    label=f"Chi-squared PDF (df={2 * acq_config.num_blocks})",
)
ax.vlines([acq_result.detection_threshold], ymin=0, ymax=np.max(hist), color="black", linestyle="--", label="Detection Threshold")
ax.legend()
ax.set_yscale("log")
ax.set_ylim(1, 1e6)
ax.set_title(f"Histogram of Correlation Magnitudes for {sig_id}")
ax.set_xlabel("Correlation Magnitude")
ax.set_ylabel("Count")
ax.grid()
ax.set_xlim(0, 120)
plt.show()

In [ ]:
# Save acquisition results to file
import pickle
acq_results_directory = os.path.join(local_data_dir, "acquisition-results")
os.makedirs(acq_results_directory, exist_ok=True)
acq_results_version_id = "v1"
acq_results_filepath = os.path.join(
    acq_results_directory, f"{collect_id}.{acq_results_version_id}.pkl")
with open(acq_results_filepath, "wb") as f:
    pickle.dump(acq_results, f)

In [ ]:
acq_result.peak_code_phase_bin